In [ ]:
import os, copy, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from google.colab import drive

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/AML_Dataset/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

!pip install timm -q
import timm

Mounted at /content/drive


In [ ]:
from google.colab import files
uploaded = files.upload()

import zipfile
with zipfile.ZipFile("WaRP-C-preprocessed.zip", "r") as z:
    z.extractall("/content/WaRP-C-preprocessed")

Saving WaRP-C-preprocessed.zip to WaRP-C-preprocessed.zip


In [ ]:
PREPROCESSED_ROOT = "/content/WaRP-C-preprocessed"
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

CFG = {
    "num_classes": 28,
    "model_name": "dm_nfnet_f0",
    "lr": 3e-4,
    "min_lr":1e-6,
    "weight_decay":0.05,
    "label_smoothing":0.1,
    "focal_gamma":2.0,
    "warmup_epochs": 3,
    "num_epochs":15,
    "blend_alpha":0.4,
    "early_stop_patience":5,
}

full_train_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
])

eval_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
#prepare dataset
def _make_flat_dataset(root_dir, transform):
    samples, class_to_idx = [], {}
    for superclass in sorted(os.listdir(root_dir)):
        sp = os.path.join(root_dir, superclass)
        if not os.path.isdir(sp): continue
        for subclass in sorted(os.listdir(sp)):
            scp = os.path.join(sp, subclass)
            if not os.path.isdir(scp): continue
            if subclass not in class_to_idx:
                class_to_idx[subclass] = len(class_to_idx)
            for img_name in os.listdir(scp):
                if img_name.lower().endswith(".jpg"):
                    samples.append((os.path.join(scp, img_name), class_to_idx[subclass]))
    dataset = datasets.ImageFolder(root_dir, transform=transform)
    dataset.samples = dataset.imgs = samples
    dataset.targets = [s[1] for s in samples]
    dataset.classes = list(class_to_idx.keys())
    dataset.class_to_idx = class_to_idx
    return dataset


def get_dataloaders(root=PREPROCESSED_ROOT, batch_size=32, num_workers=2, seed=42):
    torch.manual_seed(seed)
    train_ds = _make_flat_dataset(f"{root}/train", transform=full_train_pipeline)
    val_ds   = _make_flat_dataset(f"{root}/val",   transform=eval_pipeline)
    test_ds  = _make_flat_dataset(f"{root}/test",  transform=eval_pipeline)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader= DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=True)
    test_loader= DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds


def get_class_counts(dataset):
    counts = np.bincount(dataset.targets, minlength=len(dataset.classes))
    return counts.astype(np.float32)


train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = get_dataloaders()
num_classes = len(train_loader.dataset.classes)
print(f"Classes: {num_classes} | Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Classes: 28 | Train: 7058 | Val: 1765 | Test: 1551


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(
            logits, targets, weight=self.weight,
            label_smoothing=self.label_smoothing, reduction="none"
        )
        pt = torch.exp(-ce_loss)
        focal_term = (1 - pt) ** self.gamma
        return (focal_term * ce_loss).mean()


def get_criterion(loss_type, samples_per_class=None, device=None, label_smoothing=0.1, gamma=2.0):
    weight = None
    if loss_type in ("weighted_ce", "focal") and samples_per_class is not None:
        freq_inverse = 1.0 / (samples_per_class + 1e-6)
        weight = torch.tensor(freq_inverse / freq_inverse.sum(), dtype=torch.float).to(device)

    if loss_type == "ce":
        return nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    elif loss_type == "weighted_ce":
        return nn.CrossEntropyLoss(weight=weight, label_smoothing=label_smoothing)
    elif loss_type == "focal":
        return FocalLoss(weight=weight, gamma=gamma, label_smoothing=label_smoothing)
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")

In [ ]:
#Build the model
def build_nfnet(num_classes, freeze_backbone=False):
    model = timm.create_model(CFG["model_name"], pretrained=True, num_classes=num_classes)
    if freeze_backbone:
        for name, param in model.named_parameters():
            if "head" not in name and "fc" not in name and "classifier" not in name:
                param.requires_grad = False
    return model


def get_param_groups(model):
    head_params, backbone_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "head" in name or "fc" in name or "classifier" in name:
            head_params.append(param)
        else:
            backbone_params.append(param)
    return backbone_params, head_params

In [ ]:
def mixup_data(x, y, alpha=1.0):
    blend_ratio = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(x.device)
    blended_imgs = blend_ratio * x + (1 - blend_ratio) * x[index, :]
    return blended_imgs, y, y[index], blend_ratio

def mixup_criterion(criterion, pred, labels_orig, labels_mixed, blend_ratio):
    return blend_ratio * criterion(pred, labels_orig) + (1 - blend_ratio) * criterion(pred, labels_mixed)

def train_one_epoch(model, loader, optimizer, criterion, blend_alpha=0.0):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if blend_alpha > 0:
            imgs, labels_orig, labels_mixed, blend_ratio = mixup_data(imgs, labels, blend_alpha)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = mixup_criterion(criterion, logits, labels_orig, labels_mixed, blend_ratio) if blend_alpha > 0 else criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, eval_device=None):
    eval_device = eval_device or device
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(eval_device), labels.to(eval_device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels

In [ ]:
#Run the model
def run_nfnet_experiment(freeze_backbone, loss_type):
    config_name = f"nfnet_frozen{freeze_backbone}_loss{loss_type}"
    checkpoint_path = f"{CHECKPOINT_DIR}/{config_name}_checkpoint.pth"

    print(f"CONFIG: {config_name}")
    model = build_nfnet(num_classes, freeze_backbone=freeze_backbone).to(device)

    samples_per_class = get_class_counts(train_ds)
    criterion = get_criterion(loss_type, samples_per_class=samples_per_class, device=device,
                               label_smoothing=CFG["label_smoothing"], gamma=CFG["focal_gamma"])

    backbone_params, head_params = get_param_groups(model)
    if freeze_backbone:
        optimizer = optim.AdamW(head_params, lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    else:
        optimizer = optim.AdamW([
            {"params": backbone_params, "lr": CFG["lr"] / 10},
            {"params": head_params, "lr": CFG["lr"]},
        ], weight_decay=CFG["weight_decay"])

    def get_lr_scale(epoch):
        if epoch < CFG["warmup_epochs"]:
            return (epoch + 1) / CFG["warmup_epochs"]
        decay_progress = (epoch - CFG["warmup_epochs"]) / max(1, CFG["num_epochs"] - CFG["warmup_epochs"])
        return CFG["min_lr"] / CFG["lr"] + 0.5 * (1 - CFG["min_lr"] / CFG["lr"]) * (1 + np.cos(np.pi * decay_progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, get_lr_scale)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc, patience_counter, start_epoch = 0.0, 0, 0

    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        start_epoch = ckpt["epoch"] + 1
        best_val_acc = ckpt["best_val_acc"]
        patience_counter = ckpt["patience_counter"]
        history = ckpt["history"]


    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(start_epoch, CFG["num_epochs"]):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, blend_alpha=CFG["blend_alpha"])
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch [{epoch+1:02d}/{CFG['num_epochs']}] "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print(f"  New best val acc: {best_val_acc:.4f}")
        else:
            patience_counter += 1

        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "best_model_state": best_state,
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_val_acc": best_val_acc,
            "patience_counter": patience_counter,
            "history": history,
            "cfg": CFG,
        }, checkpoint_path)

        if patience_counter >= CFG["early_stop_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)
    test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)
    precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average="weighted", zero_division=0)
    cm = confusion_matrix(test_labels, test_preds)

    result = {
        "freeze_backbone": freeze_backbone,
        "loss_type": loss_type,
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "history": history,
    }

    with open(f"{config_name}_results.json", "w") as f:
        json.dump({k: v for k, v in result.items()}, f, indent=2)
    np.savez(f"{config_name}_test_predictions.npz",
             preds=np.array(test_preds), labels=np.array(test_labels),
             class_names=np.array(train_loader.dataset.classes))
    torch.save(model.state_dict(), f"{config_name}_final.pth")

    from google.colab import files
    files.download(f"{config_name}_results.json")
    files.download(f"{config_name}_test_predictions.npz")
    files.download(f"{config_name}_final.pth")

    return result, model, criterion, test_loader, cm

In [ ]:
#Ablation study(Fine-tuned backbone +plain Cross entropy)
result_r1, model_r1, criterion_r1, test_loader_r1, cm_r1 = run_nfnet_experiment(freeze_backbone=False, loss_type="ce")
print(result_r1)

CONFIG: nfnet_frozenFalse_lossce


model.safetensors: reconstructing file:   0%|          |  0.00B /  286MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch [01/15] Train Loss: 2.4252 Acc: 0.279 | Val Loss: 1.6198 Acc: 0.637 | LR: 2.00e-05
  New best val acc: 0.6368
Epoch [02/15] Train Loss: 2.0170 Acc: 0.327 | Val Loss: 1.4069 Acc: 0.708 | LR: 3.00e-05
  New best val acc: 0.7082
Epoch [03/15] Train Loss: 1.8746 Acc: 0.394 | Val Loss: 1.3326 Acc: 0.747 | LR: 3.00e-05
  New best val acc: 0.7473
Epoch [04/15] Train Loss: 1.7529 Acc: 0.393 | Val Loss: 1.2997 Acc: 0.747 | LR: 2.95e-05
Epoch [05/15] Train Loss: 1.6670 Acc: 0.438 | Val Loss: 1.2656 Acc: 0.771 | LR: 2.80e-05
  New best val acc: 0.7705
Epoch [06/15] Train Loss: 1.6526 Acc: 0.415 | Val Loss: 1.2620 Acc: 0.771 | LR: 2.56e-05
Epoch [07/15] Train Loss: 1.5655 Acc: 0.451 | Val Loss: 1.1888 Acc: 0.788 | LR: 2.25e-05
  New best val acc: 0.7875
Epoch [08/15] Train Loss: 1.5301 Acc: 0.443 | Val Loss: 1.2257 Acc: 0.773 | LR: 1.89e-05
Epoch [09/15] Train Loss: 1.4207 Acc: 0.514 | Val Loss: 1.1980 Acc: 0.787 | LR: 1.50e-05
Epoch [10/15] Train Loss: 1.4006 Acc: 0.488 | Val Loss: 1.2015 A

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'freeze_backbone': False, 'loss_type': 'ce', 'best_val_acc': 0.8090651558073655, 'test_acc': 0.7975499677627337, 'precision': 0.8004575047320931, 'recall': 0.7975499677627337, 'f1': 0.7971205741145925, 'history': {'train_loss': [2.4251932566816157, 2.016982999714938, 1.8745905789462003, 1.7529226479205218, 1.6669838149439204, 1.6525725892998955, 1.5655364843932065, 1.5300545378164812, 1.4207054950974205, 1.4006484289060939, 1.3520953663370825, 1.393550749800422, 1.2606726903807033, 1.3427685298702934, 1.3947380526499316], 'val_loss': [1.6198110214373882, 1.4069279407306703, 1.3325558855918582, 1.2997004855455825, 1.2656340797967343, 1.262042234639608, 1.1888343583423084, 1.2257309615105474, 1.1979718473410133, 1.2015455224696368, 1.1958457645883642, 1.2000792886312555, 1.1696682193123922, 1.1707160068976643, 1.1738191845396762], 'train_acc': [0.2785511363636364, 0.3274147727272727, 0.39360795454545455, 0.3931818181818182, 0.43849431818181817, 0.4147727272727273, 0.4509943181818182, 0.

In [ ]:
#Ablation study(Fine-tuned backbone +focal loss)
result_r2, model_r2, criterion_r2, test_loader_r2, cm_r2 = run_nfnet_experiment(freeze_backbone=False, loss_type="focal")
print(result_r2)

CONFIG: nfnet_frozenFalse_lossfocal
Epoch [01/15] Train Loss: 0.0007 Acc: 0.012 | Val Loss: 0.0004 Acc: 0.041 | LR: 2.00e-05
  New best val acc: 0.0414
Epoch [02/15] Train Loss: 0.0005 Acc: 0.034 | Val Loss: 0.0003 Acc: 0.069 | LR: 3.00e-05
  New best val acc: 0.0686
Epoch [03/15] Train Loss: 0.0003 Acc: 0.073 | Val Loss: 0.0002 Acc: 0.186 | LR: 3.00e-05
  New best val acc: 0.1858
Epoch [04/15] Train Loss: 0.0003 Acc: 0.115 | Val Loss: 0.0002 Acc: 0.149 | LR: 2.95e-05
Epoch [05/15] Train Loss: 0.0003 Acc: 0.120 | Val Loss: 0.0002 Acc: 0.349 | LR: 2.80e-05
  New best val acc: 0.3490
Epoch [06/15] Train Loss: 0.0002 Acc: 0.143 | Val Loss: 0.0001 Acc: 0.318 | LR: 2.56e-05
Epoch [07/15] Train Loss: 0.0002 Acc: 0.140 | Val Loss: 0.0001 Acc: 0.384 | LR: 2.25e-05
  New best val acc: 0.3836
Epoch [08/15] Train Loss: 0.0003 Acc: 0.144 | Val Loss: 0.0001 Acc: 0.398 | LR: 1.89e-05
  New best val acc: 0.3983
Epoch [09/15] Train Loss: 0.0002 Acc: 0.208 | Val Loss: 0.0001 Acc: 0.454 | LR: 1.50e-05
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'freeze_backbone': False, 'loss_type': 'focal', 'best_val_acc': 0.5167138810198301, 'test_acc': 0.5183752417794971, 'precision': 0.7147227362455694, 'recall': 0.5183752417794971, 'f1': 0.5193521801021229, 'history': {'train_loss': [0.0006636667604404714, 0.0004713550719316117, 0.00031954713188249364, 0.0003017353323891505, 0.0002625661016777899, 0.00024996680655352645, 0.00023843082480280744, 0.00025169974841529353, 0.0001860217339774615, 0.0002148106357708457, 0.00019236511847835075, 0.00017811417234373617, 0.00018798437155386307, 0.00016257727248475103, 0.00017737174837913533], 'val_loss': [0.00037074192447215845, 0.00031033183538543575, 0.00018166992342575762, 0.0001820652948071779, 0.0001532651468594994, 0.00014016347001593628, 0.00012005785525776116, 0.00012713735907299975, 0.00011565150488507683, 0.00011240234108781575, 0.00012198613324654738, 0.00011966069944567482, 0.0001314911671143527, 0.00012413632464983464, 0.00012701909474380647], 'train_acc': [0.011931818181818182, 0.033

In [ ]:
print("\nSUMMARY")
for r in [result_r1, result_r2]:
    print(f"frozen={r['freeze_backbone']:<5} loss={r['loss_type']:<12} "
          f"| acc={r['test_acc']:.4f} prec={r['precision']:.4f} rec={r['recall']:.4f} f1={r['f1']:.4f}")


SUMMARY
frozen=0     loss=ce           | acc=0.7975 prec=0.8005 rec=0.7975 f1=0.7971
frozen=0     loss=focal        | acc=0.5184 prec=0.7147 rec=0.5184 f1=0.5194
